In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# 1. Iniciamos Spark
spark = SparkSession.builder.appName("Semana13_Regresion_AgroTech").getOrCreate()

# 2. Leemos los datos etiquetados de Semana 10
ruta_datos = "/home/jovyan/work/semanas/Semana 10/modelos/datos_etiquetados_kmeans"
df_clusters = spark.read.parquet(ruta_datos)

# 3. Comprobamos columnas disponibles
print("[INFO] Columnas disponibles:", df_clusters.columns)
df_clusters.select("marca", "precio_kg", "rating", "prediction").show(10)

# ============================================================
# PREPARACIÓN DE VARIABLES PARA REGRESIÓN
# ============================================================

# A. VectorAssembler con variables predictoras
assembler_regresion = VectorAssembler(
    inputCols=["rating", "opiniones"],  # variables explicativas
    outputCol="features_regresion"
)
df_vector_reg = assembler_regresion.transform(df_clusters)

# B. Escalado de características
scaler_reg = StandardScaler(inputCol="features_regresion", outputCol="scaledFeatures_regresion")
scaler_model_reg = scaler_reg.fit(df_vector_reg)
df_para_regresion = scaler_model_reg.transform(df_vector_reg)

# C. Renombramos la variable objetivo
df_para_regresion = df_para_regresion.withColumnRenamed("precio_kg", "label_precio")

# D. Eliminamos columna prediction para evitar conflicto
if "prediction" in df_para_regresion.columns:
    df_para_regresion = df_para_regresion.drop("prediction")

# ============================================================
# DIVISIÓN EN TRAIN Y TEST
# ============================================================
train_reg, test_reg = df_para_regresion.randomSplit([0.7, 0.3], seed=42)

# ============================================================
# REGRESIÓN LINEAL
# ============================================================
lr_regresion = LinearRegression(
    featuresCol="scaledFeatures_regresion",
    labelCol="label_precio",
    maxIter=10
)

# Entrenamos el modelo
lr_reg_model = lr_regresion.fit(train_reg)

# Predicciones
predictions_regresion = lr_reg_model.transform(test_reg)

print("=== COMPARATIVA: PRECIO REAL VS PRECIO PREDICHO ===")
predictions_regresion.select("marca", "label_precio", "prediction").show(10)

# ============================================================
# EVALUACIÓN DEL MODELO
# ============================================================
evaluator_r2 = RegressionEvaluator(labelCol="label_precio", predictionCol="prediction", metricName="r2")
evaluator_rmse = RegressionEvaluator(labelCol="label_precio", predictionCol="prediction", metricName="rmse")

r2 = evaluator_r2.evaluate(predictions_regresion)
rmse = evaluator_rmse.evaluate(predictions_regresion)

print("==================================================")
print("     EVALUACIÓN DE LA REGRESIÓN (SEMANA 13)       ")
print("==================================================")
print(f"R² (Coeficiente de Determinación): {r2 * 100:.2f}%")
print(f"RMSE (Error promedio del modelo):  {rmse:.4f}")
print("==================================================")

# ============================================================
# COEFICIENTES DEL MODELO
# ============================================================
print(f"Intersección (Precio base): {lr_reg_model.intercept:.4f}")
for idx, coef in enumerate(lr_reg_model.coefficients):
    print(f"Coeficiente {idx}: {coef:.4f}")

[INFO] Columnas disponibles: ['marca', 'precio_raw', 'rating', 'opiniones', 'precio_kg', 'features', 'scaledFeatures', 'prediction']
+-----+-----------------+-----------------+----------+
|marca|        precio_kg|           rating|prediction|
+-----+-----------------+-----------------+----------+
|    0|9.020000457763672|4.800000190734863|         1|
|    1|6.860000133514404|4.699999809265137|         0|
|    3|5.150000095367432|4.699999809265137|         0|
|    1|5.349999904632568|4.699999809265137|         0|
|    0|7.639999866485596|4.800000190734863|         1|
|    1|9.359999656677246|4.900000095367432|         0|
|    3|4.670000076293945|4.800000190734863|         0|
|    0|7.510000228881836|4.800000190734863|         1|
|    3|4.670000076293945|4.699999809265137|         0|
|    0|8.739999771118164|4.800000190734863|         1|
+-----+-----------------+-----------------+----------+
only showing top 10 rows

=== COMPARATIVA: PRECIO REAL VS PRECIO PREDICHO ===
+-----+------------